In [1]:
from pyspark.sql.functions import current_timestamp, lit
def add_ingestion_date(input_df, file_date):
   output_df = input_df.withColumn("ingestion_date",current_timestamp()).withColumn("file_date",lit(file_date))
   return output_df

StatementMeta(, 44eb3d69-dc90-4ccb-bfbb-803e39942986, 3, Finished, Available, Finished, False)

In [3]:
from delta.tables import DeltaTable

def merge_delta_lake(input_df, lh_gold_name, schema_name, table_name, gold_folder_path, merge_condition, partition_column):

    from delta.tables import DeltaTable

    if (spark._jsparkSession.catalog().tableExists(f"{lh_gold_name}.{schema_name}.{table_name}")):

        deltaTablePeople = DeltaTable.forPath(spark, f'{gold_folder_path}/{schema_name}/{table_name}')

        deltaTablePeople.alias('tgt') \
            .merge(
                input_df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

    else:
        input_df.write.format("delta") \
            .mode("overwrite") \
            .partitionBy(partition_column) \
            .saveAsTable(table_name)


StatementMeta(, 44eb3d69-dc90-4ccb-bfbb-803e39942986, 5, Finished, Available, Finished, False)